# 1.7 - Ensemble Methods

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds the three families of ensemble learning by hand with scikit-learn: bagging (Random Forest), boosting (Gradient Boosting), and combining diverse models (Voting and Stacking). Every block trains on the same synthetic 20-feature classification set so you can read the accuracy differences straight off the print statements. By the end you have seen, in code, the classical-ML idea that model routing in the GenAI era is really just stacking wearing a new name.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the two libraries every block below leans on. Run this once per environment, then skip it on later runs.

In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q scikit-learn datasets


**Explanation**

A one-time environment prep cell - it installs packages and produces no model output.

**How the code works - step by step**
- **`!pip install -q scikit-learn datasets`** - pulls scikit-learn (all the ensemble estimators, the synthetic dataset generator, and the accuracy metric) plus the `datasets` helper; `-q` keeps pip quiet.

**In one line:** get scikit-learn on the machine so the rest of the notebook can import it.

**What you'll see:** (no output - environment prep)

## 1 - Bagging with a Random Forest

Bagging trains many trees in parallel, each on a random bootstrap sample of the rows and a random subset of the features, then aggregates their votes. Building 500 such trees is the fastest way to see why the ensemble is steadier than any single tree - and why its `feature_importances_` are the first thing regulators ask for.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                            n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
print(f"RF Accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}")
print(f"Top-3 features: {rf.feature_importances_.argsort()[-3:][::-1]}")


**Explanation**

This cell manufactures a labelled dataset, splits it, and fits a 500-tree Random Forest, then reports test accuracy and the three most important features. It is a self-contained train-and-measure block: the forest's diversity comes from bootstrapping rows and sampling `sqrt(features)` at each split.

**How the code works - step by step**
- **`make_classification`** - generates 2000 rows, 20 features (12 of them actually informative), a fixed `random_state` so results are reproducible.
- **`train_test_split`** - holds out 20% of the data as an untouched test set.
- **`RandomForestClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1)`** - builds 500 trees, each considering only `sqrt(20)` features per split, trained across all CPU cores.
- **`rf.fit` then `accuracy_score(y_te, rf.predict(X_te))`** - trains on the training split, scores on the held-out test split.
- **`rf.feature_importances_.argsort()[-3:][::-1]`** - ranks features by importance and prints the top three indices, highest first.

**In one line:** bootstrap many trees in parallel, average their votes, then read off which features drove the decision.

**What you'll see:** Two lines: a test accuracy such as `RF Accuracy: 0.9xxx`, and `Top-3 features:` followed by three feature indices ordered most-important first.

## 2 - Boosting with Gradient Boosting

Boosting flips bagging's parallelism on its head: trees are built one after another, each new tree trained to correct the errors the ensemble has made so far. This block uses scikit-learn's `GradientBoostingClassifier` (the XGBoost-style idea) and times the fit, because sequential training is the price you pay for boosting's usual accuracy edge on tabular data.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# sklearn's GradientBoosting (XGBoost-like behavior)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                max_depth=3, random_state=42)
start = time.time()
gb.fit(X_tr, y_tr)
elapsed = time.time() - start
acc = accuracy_score(y_te, gb.predict(X_te))
print(f"GradientBoosting: accuracy={acc:.4f}, time={elapsed:.2f}s")


**Explanation**

This cell rebuilds the identical dataset and split, then fits 200 boosted trees while wrapping the fit in a timer. It is a measurement harness as much as a model - it prints both accuracy and wall-clock training time so you can weigh the sequential cost against the accuracy.

**How the code works - step by step**
- **`make_classification` / `train_test_split`** - regenerates the same 2000-row, 20-feature problem with the same `random_state`, so its numbers are comparable to the Random Forest above.
- **`GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3)`** - 200 shallow trees added in sequence; `learning_rate=0.1` shrinks each tree's contribution so no single tree dominates.
- **`start = time.time()` around `gb.fit`** - measures how long the sequential training actually takes.
- **`accuracy_score(y_te, gb.predict(X_te))`** - scores the boosted ensemble on the held-out test set.

**In one line:** add shallow trees one at a time, each fixing the last ensemble's mistakes, and time how long that costs.

**What you'll see:** A single line like `GradientBoosting: accuracy=0.9xxx, time=X.XXs` - the test accuracy plus the seconds the sequential fit took.

## 3 - Combining diverse models: Voting vs Stacking

The final step stops tuning one algorithm and instead blends three different ones - a Random Forest, a Gradient Boosting model, and an SVM. Voting simply averages their probabilities; Stacking trains a small meta-model to learn which base model to trust for which input. That meta-learner is the classical blueprint behind LLM routing.

In [ ]:
from sklearn.ensemble import (StackingClassifier, VotingClassifier,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Base models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
svc = SVC(probability=True, random_state=42)

# Voting: all models vote
voter = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svc', svc)],
    voting='soft'  # average probabilities
)

# Stacking: meta-learner decides
stacker = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svc', svc)],
    final_estimator=LogisticRegression(),
    cv=5  # out-of-fold predictions
)

voter.fit(X_tr, y_tr)
stacker.fit(X_tr, y_tr)
print(f"Voting   accuracy: {accuracy_score(y_te, voter.predict(X_te)):.4f}")
print(f"Stacking accuracy: {accuracy_score(y_te, stacker.predict(X_te)):.4f}")


**Explanation**

This cell wires the same three base estimators into two different combiners and fits both on the shared dataset. It is a side-by-side comparison: `VotingClassifier` does a fixed soft-probability average, while `StackingClassifier` fits a `LogisticRegression` on the base models' out-of-fold predictions, so the two printed accuracies show what the learned meta-layer buys you over a plain average.

**How the code works - step by step**
- **Base models** - `rf` (100-tree Random Forest), `gb` (100-tree Gradient Boosting), and `svc` (`SVC(probability=True)` so it can emit probabilities).
- **`VotingClassifier(..., voting='soft')`** - averages the three models' predicted probabilities and picks the top class; no meta-model, just a mean.
- **`StackingClassifier(..., final_estimator=LogisticRegression(), cv=5)`** - generates 5-fold out-of-fold predictions from the base models, then trains a logistic-regression meta-learner on them to weigh each base model per input.
- **`voter.fit` / `stacker.fit` then two `accuracy_score` prints** - trains both combiners on the training split and reports each one's test accuracy.

**In one line:** average three diverse models (voting) versus letting a meta-learner decide whom to trust (stacking), then compare.

**What you'll see:** Two lines, `Voting   accuracy: 0.9xxx` and `Stacking accuracy: 0.9xxx`, letting you compare the fixed average against the learned meta-learner on the same test set.

Across four cells you moved through the whole ensemble ladder: bagging trains many trees in parallel on bootstrapped samples and lets them vote; boosting trains trees sequentially so each one fixes the last one's mistakes; and voting/stacking blend different model families, with stacking adding a meta-learner that decides whom to trust per input. The same synthetic dataset ran through all three so the accuracy numbers are directly comparable. Carry the stacking intuition forward - the meta-learner routing between base models is the classical ancestor of the LLM router that picks which model answers which prompt in later modules.